# Solana Investigation V2

This duplicate notebook keeps the original Solana notebook untouched and uses the upgraded `solana_second_level_v2.py` pipeline instead. It works over the **last 3 days** at **1-second resolution**. The charts are arranged to answer: **do opportunities exist, at what size, and are they still positive under stricter fee assumptions?**

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from solana_second_level_v2 import MARKET_SPEC, dataset_summary_frame, load_or_build_solana_dataset

plt.style.use('default')

REFRESH_SOLANA_DATA = False
market_dataset = load_or_build_solana_dataset(project_root=PROJECT_ROOT, refresh=REFRESH_SOLANA_DATA, verbose=True)
manifest = market_dataset['manifest']
size_sensitivity = market_dataset['size_sensitivity'].copy().sort_values('trade_size_base').reset_index(drop=True)
arb_labels = market_dataset['arb_labels'].copy().sort_values('timestamp').reset_index(drop=True)
opportunity_windows = market_dataset['opportunity_windows'].copy()

summary_rows = [
    {'field': 'chain_name', 'value': MARKET_SPEC.chain_name},
    {'field': 'pair_label', 'value': MARKET_SPEC.pair_label},
    {'field': 'market_slug', 'value': MARKET_SPEC.slug},
    {'field': 'sample_window_days', 'value': MARKET_SPEC.sample_window_days},
    {'field': 'state_frequency', 'value': MARKET_SPEC.frequency},
    {'field': 'window_start_utc', 'value': manifest['window_start_utc']},
    {'field': 'window_end_utc', 'value': manifest['window_end_utc']},
    {'field': 'primary_trade_size_base', 'value': MARKET_SPEC.primary_trade_size_base},
    {'field': 'trade_sizes_base', 'value': ', '.join(str(x) for x in MARKET_SPEC.trade_sizes_base)},
    {'field': 'network_fee_lamports_floor', 'value': manifest['network_fee_bounds']['network_fee_lamports_floor']},
    {'field': 'network_fee_lamports_p95', 'value': manifest['network_fee_bounds']['network_fee_lamports_p95']},
    {'field': 'positive_network_floor_seconds', 'value': manifest['positive_network_floor_seconds']},
    {'field': 'positive_network_p95_seconds', 'value': manifest['positive_network_p95_seconds']},
]

primary_row = size_sensitivity.loc[np.isclose(size_sensitivity['trade_size_base'], MARKET_SPEC.primary_trade_size_base)].iloc[0]
largest_floor_size = size_sensitivity.loc[size_sensitivity['network_floor_positive_seconds'] > 0, 'trade_size_base'].max() if (size_sensitivity['network_floor_positive_seconds'] > 0).any() else np.nan
largest_p95_size = size_sensitivity.loc[size_sensitivity['network_p95_positive_seconds'] > 0, 'trade_size_base'].max() if (size_sensitivity['network_p95_positive_seconds'] > 0).any() else np.nan
primary_floor_rate = (primary_row['network_floor_positive_seconds'] / primary_row['observed_seconds'] * 100.0) if primary_row['observed_seconds'] else 0.0
primary_p95_rate = (primary_row['network_p95_positive_seconds'] / primary_row['observed_seconds'] * 100.0) if primary_row['observed_seconds'] else 0.0
if primary_row['network_p95_positive_seconds'] > 0:
    verdict = 'Positive under both fee floor and p95 network-fee bands, but still only a lower-bound Solana screen.'
elif primary_row['network_floor_positive_seconds'] > 0:
    verdict = 'Positive only under the fee floor; not robust under the stricter p95 network-fee band.'
else:
    verdict = 'No positive seconds at the primary size.'

verdict_rows = [
    {'field': 'verdict', 'value': verdict},
    {'field': 'primary_size_floor_positive_seconds', 'value': int(primary_row['network_floor_positive_seconds'])},
    {'field': 'primary_size_p95_positive_seconds', 'value': int(primary_row['network_p95_positive_seconds'])},
    {'field': 'primary_size_floor_positive_rate_pct', 'value': round(primary_floor_rate, 3)},
    {'field': 'primary_size_p95_positive_rate_pct', 'value': round(primary_p95_rate, 3)},
    {'field': 'largest_size_with_floor_positives', 'value': largest_floor_size},
    {'field': 'largest_size_with_p95_positives', 'value': largest_p95_size},
]

display(pd.DataFrame(summary_rows))
display(dataset_summary_frame(market_dataset))
display(Markdown('## Opportunity Verdict'))
display(pd.DataFrame(verdict_rows))
display(Markdown('## Selected Pools'))
display(pd.DataFrame(market_dataset['selected_pools']))
display(Markdown('## Fee Bounds Used In The Lower-Bound Profit Screen'))
display(pd.DataFrame([market_dataset['fee_bounds']]))
display(Markdown('## QC Report'))
display(pd.DataFrame(market_dataset['qc_report'].items(), columns=['check', 'value']))


In [ ]:
venue_columns = [
    'dex', 'source', 'pool_address', 'pool_type', 'tvl_usd', 'volume_24h_usd',
    'fee_rate_floor', 'fee_model', 'reconstructable', 'reconstructable_reason'
]
venue_catalog = market_dataset['venue_catalog'].copy()

display(Markdown('## Opportunity Tables'))
display(Markdown('### Official Venue Diagnosis'))
display(venue_catalog.loc[:, [column for column in venue_columns if column in venue_catalog.columns]])

display(Markdown('### Size Sensitivity'))
display(size_sensitivity)

top_columns = [
    'timestamp', 'buy_dex', 'sell_dex', 'trade_size_base', 'gross_edge_quote', 'gross_edge_bps',
    'edge_after_pool_fees_quote_floor', 'network_fee_quote_floor', 'network_fee_quote_p95',
    'net_edge_quote_floor', 'net_edge_quote_p95', 'net_edge_bps_floor', 'net_edge_bps_p95',
    'positive_after_pool_fees_floor', 'positive_after_network_floor', 'positive_after_network_p95', 'stale_state'
]
display(Markdown('### Top Seconds'))
display(arb_labels.sort_values('net_edge_bps_floor', ascending=False).head(25).loc[:, [c for c in top_columns if c in arb_labels.columns]])

display(Markdown('### Positive Windows (Network Fee Floor)'))
display(opportunity_windows.head(25))

display(Markdown('### Positive Counts By Direction'))
direction_counts = (
    arb_labels.groupby(['buy_dex', 'sell_dex'], as_index=False)
    .agg(
        observed_seconds=('timestamp', 'count'),
        non_stale_seconds=('stale_state', lambda s: int((~s).sum())),
        gross_positive_seconds=('gross_edge_quote', lambda s: int((s > 0).sum())),
        pool_fee_floor_positive_seconds=('positive_after_pool_fees_floor', 'sum'),
        network_floor_positive_seconds=('positive_after_network_floor', 'sum'),
        network_p95_positive_seconds=('positive_after_network_p95', 'sum'),
        max_net_edge_bps_floor=('net_edge_bps_floor', 'max'),
        max_net_profit_quote_floor=('net_edge_quote_floor', 'max'),
    )
    .sort_values(['network_p95_positive_seconds', 'network_floor_positive_seconds', 'max_net_edge_bps_floor'], ascending=[False, False, False])
    .reset_index(drop=True)
)
direction_counts['floor_positive_rate_pct'] = np.where(direction_counts['non_stale_seconds'] > 0, direction_counts['network_floor_positive_seconds'] / direction_counts['non_stale_seconds'] * 100.0, np.nan)
direction_counts['p95_positive_rate_pct'] = np.where(direction_counts['non_stale_seconds'] > 0, direction_counts['network_p95_positive_seconds'] / direction_counts['non_stale_seconds'] * 100.0, np.nan)
display(direction_counts)


In [ ]:
pool_state = market_dataset['pool_state'].copy().sort_values('timestamp').reset_index(drop=True)

display(Markdown('## Opportunity Dashboard'))

size_plot = size_sensitivity.copy()
size_plot['gross_positive_rate_pct'] = np.where(size_plot['observed_seconds'] > 0, size_plot['gross_positive_seconds'] / size_plot['observed_seconds'] * 100.0, np.nan)
size_plot['pool_fee_rate_pct'] = np.where(size_plot['observed_seconds'] > 0, size_plot['pool_fee_floor_positive_seconds'] / size_plot['observed_seconds'] * 100.0, np.nan)
size_plot['network_floor_rate_pct'] = np.where(size_plot['observed_seconds'] > 0, size_plot['network_floor_positive_seconds'] / size_plot['observed_seconds'] * 100.0, np.nan)
size_plot['network_p95_rate_pct'] = np.where(size_plot['observed_seconds'] > 0, size_plot['network_p95_positive_seconds'] / size_plot['observed_seconds'] * 100.0, np.nan)
size_plot['trade_size_label'] = size_plot['trade_size_base'].map(lambda x: f'{x:g}')

hourly = (
    arb_labels.assign(
        hour=arb_labels['timestamp'].dt.floor('1h'),
        non_stale=(~arb_labels['stale_state']).astype(int),
        floor_positive=arb_labels['positive_after_network_floor'].astype(int),
        p95_positive=arb_labels['positive_after_network_p95'].astype(int),
    )
    .groupby('hour', as_index=False)
    .agg(
        non_stale_seconds=('non_stale', 'sum'),
        floor_positive_seconds=('floor_positive', 'sum'),
        p95_positive_seconds=('p95_positive', 'sum'),
        max_net_edge_bps_floor=('net_edge_bps_floor', 'max'),
    )
)
hourly['floor_positive_rate_pct'] = np.where(hourly['non_stale_seconds'] > 0, hourly['floor_positive_seconds'] / hourly['non_stale_seconds'] * 100.0, np.nan)
hourly['p95_positive_rate_pct'] = np.where(hourly['non_stale_seconds'] > 0, hourly['p95_positive_seconds'] / hourly['non_stale_seconds'] * 100.0, np.nan)

if not opportunity_windows.empty:
    best_window = opportunity_windows.iloc[0]
    focus_mask = (arb_labels['timestamp'] >= best_window['start_timestamp']) & (arb_labels['timestamp'] <= best_window['end_timestamp'])
    focus = arb_labels.loc[focus_mask].sort_values('timestamp')
    focus_title = f"Best positive window: {best_window['seconds']} seconds"
else:
    focus = arb_labels.nlargest(min(900, len(arb_labels)), 'net_edge_bps_floor').sort_values('timestamp')
    focus_title = 'Top scored seconds by lower-bound net edge'

fig, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)

axes[0, 0].plot(size_plot['trade_size_label'], size_plot['gross_positive_rate_pct'], marker='o', linewidth=2, label='Gross positive share')
axes[0, 0].plot(size_plot['trade_size_label'], size_plot['pool_fee_rate_pct'], marker='o', linewidth=2, label='After pool fees')
axes[0, 0].plot(size_plot['trade_size_label'], size_plot['network_floor_rate_pct'], marker='o', linewidth=2, label='After network fee floor')
axes[0, 0].plot(size_plot['trade_size_label'], size_plot['network_p95_rate_pct'], marker='o', linewidth=2, label='After network fee p95')
axes[0, 0].set_title('Positive share by trade size')
axes[0, 0].set_ylabel('% of observed seconds')
axes[0, 0].set_xlabel('Trade size (base token)')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

axes[0, 1].bar(size_plot['trade_size_label'], size_plot['max_net_edge_bps_floor'], alpha=0.8, label='Max net edge floor (bps)')
axes[0, 1].plot(size_plot['trade_size_label'], size_plot['mean_net_edge_bps_floor'], color='black', marker='o', linewidth=1.8, label='Mean net edge floor (bps)')
axes[0, 1].axhline(0.0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Lower-bound edge by trade size')
axes[0, 1].set_ylabel('Basis points')
axes[0, 1].set_xlabel('Trade size (base token)')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(hourly['hour'], hourly['floor_positive_seconds'], marker='o', linewidth=1.8, label='Floor positive seconds')
axes[1, 0].plot(hourly['hour'], hourly['p95_positive_seconds'], marker='o', linewidth=1.8, label='P95 positive seconds')
hourly_rate_ax = axes[1, 0].twinx()
hourly_rate_ax.plot(hourly['hour'], hourly['floor_positive_rate_pct'], color='tab:orange', linewidth=1.6, label='Floor positive rate (%)')
hourly_rate_ax.plot(hourly['hour'], hourly['p95_positive_rate_pct'], color='tab:red', linewidth=1.6, linestyle='--', label='P95 positive rate (%)')
axes[1, 0].set_title('Hourly clustering of positive seconds')
axes[1, 0].set_ylabel('Positive seconds')
hourly_rate_ax.set_ylabel('Positive rate (%)')
axes[1, 0].grid(alpha=0.3)
lines_a, labels_a = axes[1, 0].get_legend_handles_labels()
lines_b, labels_b = hourly_rate_ax.get_legend_handles_labels()
axes[1, 0].legend(lines_a + lines_b, labels_a + labels_b, loc='upper right')

axes[1, 1].plot(focus['timestamp'], focus['net_edge_bps_floor'], linewidth=1.8, label='Net edge floor (bps)')
if 'net_edge_bps_p95' in focus.columns:
    axes[1, 1].plot(focus['timestamp'], focus['net_edge_bps_p95'], linewidth=1.4, linestyle='--', label='Net edge p95 (bps)')
axes[1, 1].fill_between(focus['timestamp'], 0, focus['net_edge_bps_floor'], where=focus['positive_after_network_floor'].to_numpy(), alpha=0.25, color='tab:green', label='Floor-positive seconds')
axes[1, 1].axhline(0.0, color='red', linestyle='--', linewidth=1)
axes[1, 1].set_title(focus_title)
axes[1, 1].set_ylabel('Basis points')
axes[1, 1].set_xlabel('Timestamp (UTC)')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.show()

display(Markdown('## Price Context'))
price_plot = (
    pool_state.assign(plot_timestamp=pool_state['timestamp'].dt.floor('5min'))
    .groupby(['plot_timestamp', 'dex'], as_index=False)
    .agg(mid_price_quote_per_base=('mid_price_quote_per_base', 'last'))
)
fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
for dex, group in price_plot.groupby('dex'):
    ax.plot(group['plot_timestamp'], group['mid_price_quote_per_base'], label=dex, linewidth=1.5)
ax.set_title('5-minute sampled venue prices')
ax.set_ylabel('USDC per SOL')
ax.set_xlabel('Timestamp (UTC)')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
manifest_path = PROJECT_ROOT / 'outputs' / MARKET_SPEC.slug / 'metadata' / 'dataset_manifest.json'
display(Markdown(f'## Manifest\n\nSaved manifest: `{manifest_path}`'))
if manifest_path.exists():
    manifest_payload = json.loads(manifest_path.read_text())
    display(pd.json_normalize(manifest_payload, sep='.', max_level=2).T.reset_index().rename(columns={'index': 'field', 0: 'value'}))
